In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,141,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,141,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,141,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,141,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,zero,0,129,0.0
539996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,zero,0,129,0.0
539997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,zero,0,129,0.0
539998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,zero,0,129,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  465367.222202
                                  2                  425061.778950
                                  3                  403525.660917
                                  4                  367138.246411
                                  5                  271689.058401
intervention  maternal_disorders  1                  448065.369465
                                  2                  402784.924398
                                  3                  385532.028562
                                  4                  348956.937569
                                  5                  257855.853923
zero          maternal_disorders  1                  465367.222202
                                  2                  425061.778950
                                  3                  403525.660917
                                  4                  367138.246411
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,141,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,141,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,141,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,141,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1889995,ylds,cause,pregnancy,postpartum,95_plus,severe,5,zero,0,129,0.0
1889996,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,zero,0,129,0.0
1889997,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,zero,0,129,0.0
1889998,ylds,cause,all_causes,all_causes,95_plus,severe,5,zero,0,129,0.0


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  46889.395443
                                  2                  50262.333276
                                  3                  37253.166732
                                  4                  40209.684127
                                  5                  19336.364380
              maternal_disorders  1                  25109.567224
                                  2                  20461.104609
                                  3                  19736.987591
                                  4                  19611.751560
                                  5                  15917.380212
intervention  anemia              1                  38279.172227
                                  2                  41263.346471
                                  3                  29682.013316
                                  4                  33278.516737
                          

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   46889.395443
                                  2                   50262.333276
                                  3                   37253.166732
                                  4                   40209.684127
                                  5                   19336.364380
              maternal_disorders  1                  490476.789426
                                  2                  445522.883559
                                  3                  423262.648508
                                  4                  386749.997972
                                  5                  287606.438613
intervention  anemia              1                   38279.172227
                                  2                   41263.346471
                                  3                   29682.013316
                                  4                   33278.516737
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # NOTE: This else branch is for processing the Ethiopia results,
    # where no Vivarium sims were run, so the corresponding DALYs should
    # just be 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,17573.002440
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,17426.546349
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,15667.693565
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,13470.893247
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,9808.177050
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,13303.500435
23996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,14875.746207
23997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,12450.165730
23998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,10637.197153


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.550835e+07
                      2                  1.601110e+07
                      3                  1.405346e+07
                      4                  1.242667e+07
                      5                  9.448066e+06
intervention  lbwsg   1                  1.543894e+07
                      2                  1.594591e+07
                      3                  1.401163e+07
                      4                  1.239511e+07
                      5                  9.426784e+06
zero          lbwsg   1                  1.550835e+07
                      2                  1.601110e+07
                      3                  1.405346e+07
                      4                  1.242667e+07
                      5                  9.448066e+06
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,712.176299,zero
1,Female,0.0,0.019178,2,682.624088,zero
2,Female,0.0,0.019178,3,600.173392,zero
3,Female,0.0,0.019178,4,469.856554,zero
4,Female,0.0,0.019178,5,386.771854,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,271.338485,intervention
746,Male,95.0,125.000000,2,230.343262,intervention
747,Male,95.0,125.000000,3,226.344288,intervention
748,Male,95.0,125.000000,4,235.814423,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  293498.721903
                      2                  291020.166161
                      3                  267559.976280
                      4                  261900.289285
                      5                  254034.007891
intervention  anemia  1                  253853.117410
                      2                  248248.644999
                      3                  224455.725786
                      4                  217628.261923
                      5                  209674.602786
zero          anemia  1                  293498.721903
                      2                  291020.166161
                      3                  267559.976280
                      4                  261900.289285
                      5                  254034.007891
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

0.0

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  264200.611379
                      2                  265912.870200
                      3                  199183.135518
                      4                  164280.273216
                      5                  123447.547575
intervention  anemia  1                  229223.583326
                      2                  229359.333720
                      3                  166904.554617
                      4                  136312.846539
                      5                  100193.679583
zero          anemia  1                  264200.611379
                      2                  265912.870200
                      3                  199183.135518
                      4                  164280.273216
                      5                  123447.547575
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

0.0

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  959141.113475
                      2                  914373.049701
                      3                  756780.116477
                      4                  681480.932890
                      5                  594164.717234
intervention  anemia  1                  830049.675234
                      2                  782732.797835
                      3                  633903.389929
                      4                  566153.707424
                      5                  488744.756718
zero          anemia  1                  959141.113475
                      2                  914373.049701
                      3                  756780.116477
                      4                  681480.932890
                      5                  594164.717234
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  414148.385733
                      2                  423492.501283
                      3                  383377.246596
                      4                  339825.075845
                      5                  298474.679448
baseline      ntd     1                  414148.385733
                      2                  423492.501283
                      3                  383377.246596
                      4                  339825.075845
                      5                  298474.679448
intervention  ntd     1                  145389.486727
                      2                  158295.641486
                      3                  175691.140222
                      4                  175860.026689
                      5                  162222.428831
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  1.006031e+06
                                  2                  9.646354e+05
                                  3                  7.940333e+05
                                  4                  7.216906e+05
                                  5                  6.135011e+05
              lbwsg               1                  1.550835e+07
                                  2                  1.601110e+07
                                  3                  1.405346e+07
                                  4                  1.242667e+07
                                  5                  9.448066e+06
              maternal_disorders  1                  4.904768e+05
                                  2                  4.455229e+05
                                  3                  4.232626e+05
                                  4                  3.867500e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)